In [19]:
%pip install pandas

import pandas as pd



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
# World Bank Internet Usage data
df = pd.read_csv('./API_IT.NET.USER.ZS_DS2_en_csv_v2_21.csv', skiprows=4)

# Drop the trailing unnamed column World Bank appends
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

df


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Aruba,ABW,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,93.5425,97.1700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,16.3000,17.3000,19.600,21.6000,23.5000,25.0000,26.8000,27.8000,28.8,30.4
2,Afghanistan,AFG,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,11.0000,13.5000,16.800,17.6000,17.0485,16.5143,17.1917,17.7089,NaN,NaN
3,Africa Western and Central,AFW,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,20.9000,23.8000,25.700,28.5000,31.5000,34.7000,37.5000,38.5000,40.0,42.4
4,Angola,AGO,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,23.2000,26.0000,29.000,32.1294,36.6347,39.3876,42.0719,44.7581,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,Kosovo,XKX,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,83.8936,89.443,NaN,NaN,NaN,NaN,NaN,NaN,NaN
262,"Yemen, Rep.",YEM,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,24.5792,26.7184,NaN,NaN,13.8152,NaN,NaN,NaN,NaN,NaN
263,South Africa,ZAF,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,54.0000,56.1674,62.400,69.6969,72.1128,74.9686,75.4556,75.6592,NaN,NaN
264,Zambia,ZMB,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,10.3000,12.2000,14.300,18.7000,24.5000,32.5000,15.0000,33.0000,NaN,NaN


In [21]:
internet_usage = df.melt(id_vars=['Country Name','Country Code','Indicator Name','Indicator Code'], var_name='Year', value_name='Internet Usage (% of population)')
internet_usage['Year'] = pd.to_numeric(internet_usage['Year'], errors='coerce')
internet_usage.drop(columns=['Indicator Name', 'Indicator Code'], inplace=True)
internet_usage

,Country Name,Country Code,Year,Internet Usage (% of population)
0,Aruba,ABW,1960,NaN
1,Africa Eastern and Southern,AFE,1960,NaN
2,Afghanistan,AFG,1960,NaN
3,Africa Western and Central,AFW,1960,NaN
4,Angola,AGO,1960,NaN
...,...,...,...,...
17551,Kosovo,XKX,2025,NaN
17552,"Yemen, Rep.",YEM,2025,NaN
17553,South Africa,ZAF,2025,NaN
17554,Zambia,ZMB,2025,NaN


In [22]:
# Merge country metadata and create grouped DataFrames
meta_country = pd.read_csv('./Metadata_Country_API_IT.NET.USER.ZS_DS2_en_csv_v2_21.csv')
meta_country
internet_usage_meta = internet_usage.merge(meta_country[['Country Code','Region','IncomeGroup','TableName']], on='Country Code', how='left')
internet_usage_meta.rename(columns={'IncomeGroup':'Income Group'}, inplace=True)
internet_usage_meta.drop(columns=['TableName'], inplace=True)
internet_usage_meta

,Country Name,Country Code,Year,Internet Usage (% of population),Region,Income Group
0,Aruba,ABW,1960,NaN,Latin America & Caribbean,High income
1,Africa Eastern and Southern,AFE,1960,NaN,NaN,NaN
2,Afghanistan,AFG,1960,NaN,Middle East & North Africa,Low income
3,Africa Western and Central,AFW,1960,NaN,NaN,NaN
4,Angola,AGO,1960,NaN,Sub-Saharan Africa,Lower middle income
...,...,...,...,...,...,...
17551,Kosovo,XKX,2025,NaN,Europe & Central Asia,Upper middle income
17552,"Yemen, Rep.",YEM,2025,NaN,Middle East & North Africa,Low income
17553,South Africa,ZAF,2025,NaN,Sub-Saharan Africa,Upper middle income
17554,Zambia,ZMB,2025,NaN,Sub-Saharan Africa,Lower middle income


In [23]:
# By-region time series (mean of internet usage percentage per year per region)

# Use the long-format dataframe for Tableau or Plotly
by_region = internet_usage_meta.groupby(['Region','Year'])['Internet Usage (% of population)'].mean().reset_index()

# Output a CSV for Tableau
by_region.to_csv('internet_usage_by_region.csv', index=False)

# Use the wide-format dataframe for Matplotlib
by_region_pivot = by_region.pivot(index='Year', columns='Region', values='Internet Usage (% of population)')

by_region


,Region,Year,Internet Usage (% of population)
0,East Asia & Pacific,1960,NaN
1,East Asia & Pacific,1961,NaN
2,East Asia & Pacific,1962,NaN
3,East Asia & Pacific,1963,NaN
4,East Asia & Pacific,1964,NaN
...,...,...,...
457,Sub-Saharan Africa,2021,37.422637
458,Sub-Saharan Africa,2022,39.237667
459,Sub-Saharan Africa,2023,41.424549
460,Sub-Saharan Africa,2024,21.963015


In [24]:
# By-income-group time series
by_income = internet_usage_meta.groupby(['Income Group','Year'])['Internet Usage (% of population)'].mean().reset_index()
by_income.to_csv('internet_usage_by_income_group.csv', index=False)

by_income_pivot = by_income.pivot(index='Year', columns='Income Group', values='Internet Usage (% of population)')
by_income_pivot.tail()


Income Group,High income,Low income,Lower middle income,Upper middle income
Year,,,,
2021,89.923649,20.221444,52.069722,75.015165
2022,90.972702,22.830079,54.134767,77.028619
2023,91.630461,24.167667,56.707038,79.204029
2024,92.721823,8.949830,59.563100,87.321562
2025,NaN,NaN,70.000000,NaN


In [25]:
# If the region is missing, it's not a country, so we can drop those entries for the country-level dataset
by_country = internet_usage_meta.dropna(subset=['Region']).reset_index(drop=True)
by_country.to_csv('internet_usage_by_country.csv', index=False)
by_country

,Country Name,Country Code,Year,Internet Usage (% of population),Region,Income Group
0,Aruba,ABW,1960,NaN,Latin America & Caribbean,High income
1,Afghanistan,AFG,1960,NaN,Middle East & North Africa,Low income
2,Angola,AGO,1960,NaN,Sub-Saharan Africa,Lower middle income
3,Albania,ALB,1960,NaN,Europe & Central Asia,Upper middle income
4,Andorra,AND,1960,NaN,Europe & Central Asia,High income
...,...,...,...,...,...,...
14317,Kosovo,XKX,2025,NaN,Europe & Central Asia,Upper middle income
14318,"Yemen, Rep.",YEM,2025,NaN,Middle East & North Africa,Low income
14319,South Africa,ZAF,2025,NaN,Sub-Saharan Africa,Upper middle income
14320,Zambia,ZMB,2025,NaN,Sub-Saharan Africa,Lower middle income
